# FraudShield: Probability Calibration & SHAP Explainability
**Project**: FraudShield – Explainable AI-Based Financial Fraud Detection & Risk Scoring
**Focus**: Reliability curves, Brier score comparison, Global beeswarm, and Local waterfall explanations.


In [ ]:
%matplotlib inline
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

sys.path.insert(0, os.path.abspath(".."))
from src.data.loader import load_raw_data
from src.data.splitter import split_data
from src.features.preprocessor import FraudPreprocessor
from src.models.train import train_xgboost, train_logistic_regression
from src.models.calibration import calibrate_model, compare_calibration
from src.explainability.shap_explainer import FraudSHAPExplainer

print("Libraries imported successfully!")


## 1. Load Data, Preprocess, and Split

In [ ]:
import yaml
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

df = load_raw_data("../data/raw/creditcard.csv")
X_train_raw, X_val_raw, X_test_raw, y_train, y_val, y_test = split_data(df, config)

preprocessor = FraudPreprocessor(config)
X_train = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)

print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")


## 2. Train Model & Calibrate Probabilities

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss

model = LogisticRegression(max_iter=500, class_weight="balanced", random_state=42)
model.fit(X_train, y_train)

calibrated_sigmoid = calibrate_model(model, X_val, y_val, method="sigmoid")
calibrated_isotonic = calibrate_model(model, X_val, y_val, method="isotonic")

brier_raw = brier_score_loss(y_val, model.predict_proba(X_val)[:, 1])
brier_sig = brier_score_loss(y_val, calibrated_sigmoid.predict_proba(X_val)[:, 1])
brier_iso = brier_score_loss(y_val, calibrated_isotonic.predict_proba(X_val)[:, 1])

print(f"Raw Model Brier Score:       {brier_raw:.5f}")
print(f"Sigmoid Calibrated Brier:   {brier_sig:.5f}")
print(f"Isotonic Calibrated Brier:  {brier_iso:.5f}")


## 3. Calibration Curve / Reliability Diagram

In [ ]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(8, 6))
prob_true_raw, prob_pred_raw = calibration_curve(y_val, model.predict_proba(X_val)[:, 1], n_bins=10)
prob_true_sig, prob_pred_sig = calibration_curve(y_val, calibrated_sigmoid.predict_proba(X_val)[:, 1], n_bins=10)

ax.plot([0, 1], [0, 1], "k--", label="Perfectly Calibrated")
ax.plot(prob_pred_raw, prob_true_raw, "s-", color="#dc3545", label=f"Uncalibrated (Brier={brier_raw:.4f})")
ax.plot(prob_pred_sig, prob_true_sig, "o-", color="#28a745", label=f"Sigmoid (Brier={brier_sig:.4f})")

ax.set_ylabel("Fraction of Positives")
ax.set_xlabel("Mean Predicted Probability")
ax.set_title("Probability Calibration Reliability Curves")
ax.legend(loc="lower right")
plt.show()


## 4. SHAP Global Feature Importance

In [ ]:
explainer = FraudSHAPExplainer(model, X_background=X_train.sample(100, random_state=42))
explainer.fit()
global_exp = explainer.explain_global(X_val.sample(200, random_state=42))

top_feats = global_exp["feature_importance_df"].head(10)
print("Top 10 Influential Features by Mean |SHAP|:")
print(top_feats)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=top_feats, x="importance", y="feature", ax=ax, palette="mako")
ax.set_title("Top 10 Global Features by Mean Absolute SHAP Value")
plt.show()


## 5. Local Waterfall Explanation for Individual Fraud Transaction

In [ ]:
# Pick a fraudulent sample from validation
fraud_idx = y_val[y_val == 1].index[0]
sample_x = X_val.loc[[fraud_idx]]
sample_pred = calibrated_sigmoid.predict_proba(sample_x)[0, 1]
print(f"Sample #{fraud_idx} Predicted Fraud Probability: {sample_pred * 100:.2f}%")

local_exp = explainer.explain_local(sample_x.iloc[0].values, preprocessor.get_feature_names())
print(f"Base expected value: {local_exp['base_value']:.4f}")
print("Top 5 local contributions:")
contrib_df = pd.DataFrame(local_exp["waterfall_data"]).sort_values(by="abs_shap", ascending=False).head(5)
print(contrib_df[["feature", "value", "shap_value"]])
